In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline
import matplotlib.pyplot as plt
import sigpy as sp
import pandas as pd
import sigpy.plot as pl
import sigpy.mri as mr
import numpy as np
import os
import jax as jx
ksp = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_ksp.npy")
coord = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_coord.npy")


import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

print(project_root)

import aymansigmri as asm

## Goal: Implement Algorithm 4.1

In [ ]:
width=4
gap = 2
num_iters = 50

resize_wid=1024
rw_div2 = int(resize_wid/2)

inner_wid=100
indiv2 = int(inner_wid/2)

inner_widnozp = 26
indiv2nozp = int(inner_widnozp/2)

### Set up ZTE style data

In [ ]:
zte_radial_kspacedlt, zte_radial_coordsdlt, zte_radial_kspacezero, zte_radial_coordszero, mask = asm.generate_zte_data(ksp=ksp, coord=coord, n_missing=(gap), undersampling_factor=1)


cartesian_zte = asm.nufft_gridding(kspace=zte_radial_kspacezero, coords=zte_radial_coordszero)

cartesian_kspace_enl = asm.zero_padding(cart_kspace=cartesian_zte, resize_x=resize_wid, resize_y=resize_wid)

inner_region, inner_mask, start, end = asm.inner_portion(enlarged_kspace=cartesian_kspace_enl, inner_sidelen=inner_wid)


im_grid0 = sp.ifft(cartesian_zte, axes=(-2, -1))
preloop = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

zte_radial_kspace1, zte_radial_coords1 = asm.generate_zte_data_OLD(ksp=ksp, coord=coord, n_missing=0, undersampling_factor=1)
cartesian_nogap = asm.nufft_gridding(kspace=zte_radial_kspace1, coords=zte_radial_coords1)

dcf1 = (zte_radial_coords1[...,0]**2 + zte_radial_coords1[...,1]**2)**0.5
im_grid3 = sp.nufft_adjoint(zte_radial_kspace1* dcf1, zte_radial_coords1)
img_nogap = np.sum(np.abs(im_grid3)**2, axis=0)**0.5

In [ ]:
inner_regionnozp, inner_masknozp, startnozp, endnozp = asm.inner_portion(enlarged_kspace=cartesian_zte, inner_sidelen=inner_widnozp)

### Build and plot mask - same as in cartesian testing

In [ ]:
cy, cx = inner_wid // 2, inner_wid // 2
r = 5
yy, xx = np.ogrid[:inner_wid, :inner_wid]
maskb = (yy - cy)**2 + (xx - cx)**2 > r**2 +2

maskb[(cx-5):(cx+6), cy] = False
maskb[(cx), (cy-5):(cy+6)] = False

inner_removedb = inner_region.copy()
inner_removedb[:,~maskb] = 0

cy, cx = inner_widnozp // 2, inner_widnozp // 2
r = 1
yy, xx = np.ogrid[:inner_widnozp, :inner_widnozp]
masknozpa = (yy - cy)**2 + (xx - cx)**2 > r**2 +2

cy, cx = inner_widnozp // 2, inner_widnozp // 2
masknozpb = np.ones((inner_widnozp, inner_widnozp), dtype=bool)

masknozpb[(cy-1):(cy+2), cx] = False
masknozpb[cy, (cx-1):(cx+2)] = False



asm.plot_mask(inner_removedb, mask=maskb, zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_wid,sidelencart=10,sidelenrad=4)
asm.plot_mask(inner_regionnozp, mask=masknozpa, zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_widnozp,sidelencart=5,sidelenrad=4)
asm.plot_mask(inner_regionnozp, mask=masknozpb, zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_widnozp,sidelencart=5,sidelenrad=4)

### Functions for inner loop and filling framework

In [ ]:
def jigsaw(output_kspace, isolation_mask, start, end, enlarged_kspace):
    processed_isolated_kspace = output_kspace.copy()
    print(enlarged_kspace.shape)
    print(isolation_mask.shape)
    recombined = enlarged_kspace * isolation_mask ### NOTE: Need to improve this
    recombined[:, start:end, start:end] = processed_isolated_kspace
    return(recombined)

def softimpute_ALS(X_H, M_H, rank, lamda, n_iters):
        I = np.eye(rank)
        m,n = np.shape(X_H)
        U = np.random.randn(m, rank) + 1j * np.random.randn(m, rank)
        V = np.random.randn(n, rank) + 1j * np.random.randn(n, rank)

        D = I.copy()


        A = np.dot(U,D)
        B = np.dot(V,D)
        iter_count = 0
        ABt = A @ B.conj().T
        
        while iter_count < n_iters:
            X_star = np.where(M_H, X_H, ABt)
            A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
            ABt = A @ B.conj().T
            X_star = np.where(M_H, X_H, ABt)
            B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)
            ABt = A @ B.conj().T
            iter_count += 1
        return(ABt)



In [ ]:
H, *_ = asm.hankel_2(inner_regionnozp, w=10,s=1)
U, S, Vh = np.linalg.svd(H, full_matrices=False)

plt.figure()
plt.semilogy(S / S[0], '.-')
plt.axhline(1e-2, ls='--', c='r')
plt.xlabel('index')
plt.ylabel('normalised singular value')

In [ ]:
H, *_ = asm.hankel_2(inner_region,w=10,s=1)
U, S, Vh = np.linalg.svd(H, full_matrices=False)

plt.figure()
plt.semilogy(S / S[0], '.-')
plt.axhline(1e-2, ls='--', c='r')
plt.xlabel('index')
plt.ylabel('normalised singular value')

## Testing Mask

In [ ]:
cy = cx = inner_widnozp // 2
yy, xx = np.ogrid[:inner_widnozp, :inner_widnozp]
d2 = (yy - cy)**2 + (xx - cx)**2

# --- circular gaps of increasing radius ---
masknozp_r1  = d2 > 1      # 5 px  (center + 4 neighbours)
masknozp_r15 = d2 > 2      # 9 px  (3x3 block)
masknozp_r2  = d2 > 4      # 13 px
masknozp_r25 = d2 > 6      # 21 px
masknozp_r3  = d2 > 9      # 29 px
masknozp_r4  = d2 > 16     # 49 px

# --- square gaps ---
def square_mask(h):                      # h = half-width, gap is (2h+1)^2
    m = np.ones((inner_widnozp, inner_widnozp), dtype=bool)
    m[cy-h:cy+h+1, cx-h:cx+h+1] = False
    return m

masknozp_sq3 = square_mask(1)   # 3x3
masknozp_sq5 = square_mask(2)   # 5x5
masknozp_sq7 = square_mask(3)   # 7x7

# --- cross / plus gaps ---
def cross_mask(h):                       # arms of half-length h
    m = np.ones((inner_widnozp, inner_widnozp), dtype=bool)
    m[cy-h:cy+h+1, cx] = False
    m[cy, cx-h:cx+h+1] = False
    return m

masknozp_cr1 = cross_mask(1)    # your masknozpb (5 px)
masknozp_cr2 = cross_mask(2)    # 9 px
masknozp_cr3 = cross_mask(3)    # 13 px

# --- single point ---
masknozp_pt = np.ones((inner_widnozp, inner_widnozp), dtype=bool)
masknozp_pt[cy, cx] = False

masks = {
    'point': masknozp_pt,'cross1': masknozp_cr1,'r1': masknozp_r1
}

for k, m in masks.items():
    print(f"{k:7s}  missing = {(~m).sum():4d}  ({100*(~m).mean():.2f}%)")

In [ ]:
window_size = 5
lam = 1e-6
rank = 10
ksp_forhankel = inner_region.copy()
num_iters = 50


results = {}
results['preloop'] = {'im': preloop} 
for name, mask in masks.items():
    im_als, output_kspace, zeroed = LORAKS_imputeals(n_iters=num_iters, window_size=window_size, cartesian_inputkspace=inner_regionnozp, dtg_mask=mask, rank= rank, lamda = lam, stride=1, cartesian_grid=cartesian_zte,inner_mask = inner_masknozp, inner_start = startnozp, inner_end=endnozp)
    results[name] = {'im': im_als}
    print("done")

results['nogap'] = {'im': img_nogap} 

ims = {f'mask: {r}': results[r]['im'] for r in results}
asm.diff_matrix(ims)

## Testing Window

In [ ]:
ksp_forhankel = inner_region.copy()
num_iters = 50
window_sizes = [5, 6, 7, 8,9,10]

results = {}
results['preloop'] = {'im': preloop}

for w in window_sizes:
    im_als, output_kspace, zeroed = LORAKS_imputeals(n_iters=num_iters, window_size=w, cartesian_inputkspace=inner_regionnozp, dtg_mask=masknozp_pt, rank=20, lamda=1*10**-6, stride=1, cartesian_grid=cartesian_zte, inner_mask=inner_masknozp, inner_start=startnozp, inner_end=endnozp)
    results[f'w {w}'] = {'im': im_als}
    print(f"done window {w}")

results['nogap'] = {'im': img_nogap}

ims = {r: results[r]['im'] for r in results}
asm.diff_matrix(ims)

## Testing lamda

In [ ]:
ksp_forhankel = inner_region.copy()
num_iters = 100
lamdas = [1e-8, 5e-7, 1e-7, 5e-6]
rank = 10
results = {}
results['preloop'] = {'im': preloop}

for lam in lamdas:
    im_als, output_kspace, zeroed = LORAKS_imputeals(n_iters=num_iters, window_size=5, cartesian_inputkspace=inner_regionnozp, dtg_mask=masknozp_pt, rank=rank, lamda=lam, stride=1, cartesian_grid=cartesian_zte, inner_mask=inner_masknozp, inner_start=startnozp, inner_end=endnozp)
    results[f'lam {lam:.0e}'] = {'im': im_als}
    print(f"done lamda {lam:.0e}")

results['nogap'] = {'im': img_nogap}

ims = {r: results[r]['im'] for r in results}
asm.diff_matrix(ims)

## Testing Ranks

In [ ]:
window_size = 5
lam = 1e-6
ksp_forhankel = inner_region.copy()
num_iters = 100
ranks = [10,12,14]

results = {}
results['preloop'] = {'im': preloop}

for rank in ranks:
    im_als, output_kspace, zeroed = LORAKS_imputeals(n_iters=num_iters, window_size=window_size, cartesian_inputkspace=inner_regionnozp, dtg_mask=masknozp_pt, rank=rank, lamda=lam, stride=1, cartesian_grid=cartesian_zte, inner_mask=inner_masknozp, inner_start=startnozp, inner_end=endnozp)
    results[f'rank {rank}'] = {'im': im_als}
    print(f"done rank {rank}")

results['nogap'] = {'im': img_nogap}

ims = {r: results[r]['im'] for r in results}
asm.diff_matrix(ims)